In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import shutil

In [3]:

# -------------------------------
# Basic Arabic Unicode Ranges
# -------------------------------
ARABIC_CHARS = r"\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF"
ARABIC_NUMS = r"\u0660-\u0669"  # ٠١٢٣٤٥٦٧٨٩
WESTERN_NUMS = r"0-9"

# -------------------------------
# Additional allowed punctuation to preserve dates, quotes, etc.
# -------------------------------
ALLOWED_PUNCT = r"\.\,\،\:\;\!\?\-\(\)\[\]\{\}\"\«\»\/"

# -------------------------------
# Main Cleaning Function
# -------------------------------
def clean_legal_text(text: str) -> str:
    # Split text into lines to preserve structure and avoid random line removals
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        # 1️⃣ Remove HTML / XML tags
        line = re.sub(r"<[^>]+>", " ", line)

        # 2️⃣ Remove page breakers ========
        line = re.sub(r"=+", " ", line)

        # 3️⃣ Remove PAGE numbers & headers (PAGE 2, PAGE 10, etc.)
        line = re.sub(r"PAGE\s*\d+", " ", line, flags=re.IGNORECASE)

        # 4️⃣ Remove English letters completely
        line = re.sub(r"[A-Za-z]", " ", line)

        # 5️⃣ Remove Quranic verses between ﴿ ﴾, but only if content is likely a verse (e.g., length > 10 chars to avoid removing short titles like ﴿مقدمـة﴾)
        # This prevents removing non-verse content enclosed in similar brackets
        def remove_verse(match):
            content = match.group(0)
            if len(content) > 20:  # Arbitrary threshold to distinguish verses from short titles
                return " "
            return content  # Keep if short (e.g., titles)
        line = re.sub(r"﴿.*?﴾", remove_verse, line, flags=re.DOTALL)

        # 6️⃣ Remove weird symbols but keep:
        # Arabic letters + Arabic numbers + Western numbers + allowed punctuation + spaces
        line = re.sub(
            rf"[^{ARABIC_CHARS}{ARABIC_NUMS}{WESTERN_NUMS}{ALLOWED_PUNCT}\s]",
            " ",
            line
        )

        # 7️⃣ Remove standalone page numbers (lines that contain only digits after cleaning)
        line = line.strip()
        if re.match(r"^\s*\d+\s*$", line):
            continue  # Skip this line entirely instead of replacing with space

        # 8️⃣ Normalize whitespace per line (but keep lines separate)
        line = re.sub(r"\s+", " ", line).strip()

        # Only add non-empty lines
        if line:
            cleaned_lines.append(line)

    # Join cleaned lines with newlines to preserve paragraph-like structure
    cleaned_text = "\n".join(cleaned_lines)

    # Optional: Spell checker (commented out)
    # You can use a library like 'pyspellchecker' or 'hunspell' if installed, but since environment has limited packages,
    # this would require additional setup. Uncomment if needed.
    # Note: Adding spell checking would increase computation time significantly for large texts,
    # as it processes word-by-word and may require dictionary lookups. For RAG prep, it's optional and
    # not necessary if text is mostly correct, as it could introduce errors in legal terms.
    # from spellchecker import SpellChecker
    # spell = SpellChecker(language='ar')  # Assuming Arabic support
    # words = cleaned_text.split()
    # corrected = [spell.correction(w) if w in spell.unknown([w]) else w for w in words]
    # cleaned_text = ' '.join(corrected)


    return cleaned_text


# If no Duplicates

In [4]:
import os
input_folder = "/content/drive/MyDrive/Graduation Project/Categories/القانون_المدني"
output_folder = "/content/drive/MyDrive/Graduation Project/القانون المدني/Cleaned"


# -------------------------------
# Batch Processing
# -------------------------------
for filename in os.listdir(input_folder):
    if filename.lower().endswith(".txt"):
        raw_path = os.path.join(input_folder, filename)

        # اسم الفايل المنظف (نفس الاسم)
        output_filename = os.path.splitext(filename)[0] + ".txt"
        output_path = os.path.join(output_folder, output_filename)

        print(f"جاري معالجة: {filename}")

        # 1️⃣ Skip if output already exists
        if os.path.exists(output_path):
            print(f"{filename} existed -> SKIP")
            print("-"*50)
            continue

        # لو مش موجودة → نعمل تنظيف
        try:
            with open(raw_path, "r", encoding="utf-8") as f:
                raw_text = f.read()

            clean_text = clean_legal_text(raw_text)

            # حفظ النسخة المنظفة
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(clean_text)

            print(f"SAVED {output_filename}")
            print("-"*50)


        except Exception as e:
            print(f"خطأ في معالجة {filename}: {e}")

print("انتهى تنظيف كل الفايلات!")

جاري معالجة: خطوات كنابة عقد البيع للمبتدئين.txt
SAVED خطوات كنابة عقد البيع للمبتدئين.txt
--------------------------------------------------
جاري معالجة: حماية_حقوق_الملكية_الفكرية_في_مجال_الانترنت.txt
SAVED حماية_حقوق_الملكية_الفكرية_في_مجال_الانترنت.txt
--------------------------------------------------
جاري معالجة: جندي عبد الملك الجزء الأول.txt
SAVED جندي عبد الملك الجزء الأول.txt
--------------------------------------------------
جاري معالجة: جندي عبد الملك الجزء الثالث.txt
SAVED جندي عبد الملك الجزء الثالث.txt
--------------------------------------------------
جاري معالجة: جندي عبد الملك الجزء الثاني.txt
SAVED جندي عبد الملك الجزء الثاني.txt
--------------------------------------------------
جاري معالجة: المواعيد_والمدد_القانونية_في_القانون_المدني.txt
SAVED المواعيد_والمدد_القانونية_في_القانون_المدني.txt
--------------------------------------------------
جاري معالجة: عقد إيجار شقة خالية طبقا للقانون رقم 4 لسنة 1996.txt
SAVED عقد إيجار شقة خالية طبقا للقانون رقم 4 لسنة 1996

# IF DUPLICATES (PREFERED)

In [5]:
import os
import shutil
# Change According to your Paths
input_folder = "/content/drive/MyDrive/Graduation Project/Categories/القانون_المدني"
output_folder = "/content/drive/MyDrive/Graduation Project/القانون المدني/Cleaned"
folder ="/content/drive/MyDrive/Graduation Project/القانون المدني"

duplicates_folder = os.path.join(folder, "duplicates")
done_folder = os.path.join(folder, "done")
same_name_diff_content_folder = os.path.join(
    folder, "same name different content"
)

os.makedirs(duplicates_folder, exist_ok=True)
os.makedirs(done_folder, exist_ok=True)
os.makedirs(same_name_diff_content_folder, exist_ok=True)

# -------------------------------
# Batch Processing
# -------------------------------
for filename in os.listdir(input_folder):
    if not filename.lower().endswith(".txt"):
        continue

    raw_path = os.path.join(input_folder, filename)
    output_path = os.path.join(output_folder, filename)

    print(f"جاري معالجة: {filename}")

    try:
        # نقرأ raw وننضف
        with open(raw_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        new_clean_text = clean_legal_text(raw_text)
        new_clean_size = len(new_clean_text.encode("utf-8"))

        # لو نفس الاسم موجود في output
        if os.path.exists(output_path):
            with open(output_path, "r", encoding="utf-8") as f:
                existing_clean_text = f.read()

            existing_clean_size = len(existing_clean_text.encode("utf-8"))

            if new_clean_size == existing_clean_size:
                # نفس المحتوى بعد التنظيف
                shutil.move(raw_path, os.path.join(duplicates_folder, filename))
                print(f"{filename} SAME CLEAN SIZE → moved to duplicates")
            else:
                # نفس الاسم لكن محتوى مختلف
                shutil.move(
                    raw_path,
                    os.path.join(same_name_diff_content_folder, filename)
                )
                print(f"{filename} SAME NAME DIFFERENT CONTENT → moved")

            print("-" * 50)
            continue

        # لو الاسم مش موجود → نحفظ عادي
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(new_clean_text)

        shutil.move(raw_path, os.path.join(done_folder, filename))

        print(f"SAVED {filename} → moved to done")
        print("-" * 50)

    except Exception as e:
        print(f"خطأ في معالجة {filename}: {e}")

print("انتهى تنظيف كل الفايلات!")


جاري معالجة: خطوات كنابة عقد البيع للمبتدئين.txt
خطوات كنابة عقد البيع للمبتدئين.txt SAME CLEAN SIZE → moved to duplicates
--------------------------------------------------
جاري معالجة: حماية_حقوق_الملكية_الفكرية_في_مجال_الانترنت.txt
حماية_حقوق_الملكية_الفكرية_في_مجال_الانترنت.txt SAME CLEAN SIZE → moved to duplicates
--------------------------------------------------
جاري معالجة: جندي عبد الملك الجزء الأول.txt
جندي عبد الملك الجزء الأول.txt SAME CLEAN SIZE → moved to duplicates
--------------------------------------------------
جاري معالجة: جندي عبد الملك الجزء الثالث.txt
جندي عبد الملك الجزء الثالث.txt SAME CLEAN SIZE → moved to duplicates
--------------------------------------------------
جاري معالجة: جندي عبد الملك الجزء الثاني.txt
جندي عبد الملك الجزء الثاني.txt SAME CLEAN SIZE → moved to duplicates
--------------------------------------------------
جاري معالجة: المواعيد_والمدد_القانونية_في_القانون_المدني.txt
المواعيد_والمدد_القانونية_في_القانون_المدني.txt SAME CLEAN SIZE → m